# 03 · Sampling — walking from noise back to data

We have a network that predicts the noise. Now we cash it in for actual samples.

> ### 📝 How this notebook works
> Cells marked **`# TODO`** are yours to write. Each is followed by a
> **self-check** cell that verifies your implementation and prints ✅.
>
> **Stuck?** The answer key is `solutions/notebooks/`, and the reference
> implementation lives in the `nanodiffusion/` package. Peeking is allowed —
> but try first.


## 1. The algorithm

Everything follows from the boxed result of notebook 02: the mean of the reverse
step, expressed through the noise. Swap the true (unknown) $\varepsilon$ for our
network's prediction $\varepsilon_\theta(x_t,t)$:

$$\mu_\theta(x_t,t)=\frac{1}{\sqrt{\alpha_t}}\Big(x_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\varepsilon_\theta(x_t,t)\Big)$$

Then a full reverse step draws from that Gaussian (reparameterization trick again):

$$\boxed{\;x_{t-1}=\mu_\theta(x_t,t)+\sqrt{\tilde\beta_t}\;z,\qquad z\sim\mathcal N(0,I)\;}$$

**The complete sampling algorithm:**

1. draw $x_T\sim\mathcal N(0,I)$ — legitimate, because $\bar\alpha_T\approx 0$
2. for $t=T, T-1, \dots, 1$: predict $\varepsilon_\theta(x_t,t)$, form $\mu_\theta$,
   add $\sqrt{\tilde\beta_t}\,z$
3. at $t=0$, return $\mu_\theta$ with **no** noise added

That's it. It's called an **ancestral sampler** because we walk back down the chain
of ancestors $x_T \to x_{T-1}\to\dots\to x_0$.

## 2. Two questions worth asking

**Why add fresh noise $z$ at every step?** It feels backwards — we're trying to
*remove* noise! But remember what we're doing: **sampling from a distribution**,
not finding a single best answer. The network's $\mu_\theta$ is only the *centre*
of the plausible $x_{t-1}$'s; the $\sqrt{\tilde\beta_t}z$ term explores that
spread. Drop it and every run from the same $x_T$ collapses toward one averaged,
over-smoothed outcome, losing diversity. (There *is* a principled way to remove the
noise and stay sharp — that's DDIM, notebook 05.)

**Why no noise on the final step?** After the $t=1\to 0$ step we want to *output*
a clean sample, not one more noisy draw. So we return the mean — our best estimate
of the clean data. Adding noise there would just smear visible grain over the
result.

**What variance to use?** We use the true posterior variance from notebook 02,
$\tilde\beta_t=\beta_t\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}$. Note it goes to
0 as $t\to0$, which is consistent with taking no noise on the last step. (DDPM
found plain $\beta_t$ works about as well.)

In [ ]:
import torch
import matplotlib.pyplot as plt

from nanodiffusion.utils import pick_device, set_seed, scatter_2d
from nanodiffusion.data import toy2d
from nanodiffusion.schedules import NoiseSchedule
from nanodiffusion.models import MLPDenoiser
from nanodiffusion.objectives import ddpm_eps_loss
from nanodiffusion.samplers import ddpm_sample as reference_sample   # for the self-check

set_seed(0)
device = pick_device()
schedule = NoiseSchedule.make("cosine", 200).to(device)
data = toy2d("swiss_roll", 8000).to(device)
print("device:", device)

### Get a trained model

We reuse the reference denoiser here so this notebook's TODO stays focused purely
on the **sampler**. Trains in a few seconds.

In [ ]:
set_seed(0)
model = MLPDenoiser().to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
for step in range(2000):
    idx = torch.randint(0, data.shape[0], (512,), device=device)
    loss = ddpm_eps_loss(model, data[idx], schedule)
    opt.zero_grad(); loss.backward(); opt.step()
model.eval()
print(f"trained, final loss {loss.item():.4f}")

## TODO — the ancestral sampler

Implement the boxed reverse step. Available on `schedule` (length-$T$ tensors,
already on the right device):

`schedule.betas` · `schedule.alphas` · `schedule.alpha_bars` ·
`schedule.alpha_bars_prev` · `schedule.sqrt_one_minus_alpha_bars`

Here we index with `[step]` (a plain int), giving a scalar per timestep — the
whole batch is at the same $t$, so no reshaping is needed.

In [ ]:
@torch.no_grad()
def my_ddpm_sample(model, schedule, shape, device, return_trajectory=False):
    '''Reverse the diffusion process: pure noise -> data.'''
    x = torch.randn(shape, device=device)        # start from x_T ~ N(0, I)
    traj = [x.clone()]

    for step in reversed(range(schedule.timesteps)):
        t = torch.full((shape[0],), step, device=device, dtype=torch.long)
        eps = model(x, t)                         # predicted noise

        beta_t = schedule.betas[step]
        alpha_t = schedule.alphas[step]
        sqrt_1m_ab = schedule.sqrt_one_minus_alpha_bars[step]

        # TODO 1: the posterior mean
        #   mean = (x - beta_t / sqrt_1m_ab * eps) / sqrt(alpha_t)
        mean = ...  # <- replace

        if step > 0:
            # TODO 2: add fresh noise scaled by sqrt of the posterior variance
            #   posterior_var = beta_t * (1 - alpha_bars_prev[step]) / (1 - alpha_bars[step])
            #   x = mean + sqrt(posterior_var) * torch.randn_like(x)
            x = ...  # <- replace
        else:
            x = mean          # final step: no noise, return the clean estimate

        if return_trajectory:
            traj.append(x.clone())

    return (x, traj) if return_trajectory else x

In [ ]:
# ---- self-check ----
# Same seed + same model => your sampler should match the reference step for step.
set_seed(123); mine = my_ddpm_sample(model, schedule, (2000, 2), device)
set_seed(123); ref = reference_sample(model, schedule, (2000, 2), device=device)
assert mine.shape == (2000, 2), f"shape {tuple(mine.shape)} != (2000, 2)"
assert torch.isfinite(mine).all(), "produced NaN/inf — check the mean formula"
assert torch.allclose(mine, ref, atol=1e-4), "doesn't match the reference sampler"

# and the generated distribution should match the real one
real_std, gen_std = data.std(0).cpu(), mine.std(0).cpu()
print(f"real std {[round(v, 3) for v in real_std.tolist()]}")
print(f"gen  std {[round(v, 3) for v in gen_std.tolist()]}")
assert torch.allclose(gen_std, real_std, atol=0.25), "distribution doesn't match the data"
print("✅ sampler correct")

## 3. Your samples

The real test is visual: does the generated cloud look like the same distribution?
These points were **never in the dataset** — the model invented them.

In [ ]:
gen = my_ddpm_sample(model, schedule, (2000, 2), device)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(8, 4))
scatter_2d(a1, data[:2000], "real data")
scatter_2d(a2, gen, "YOUR generated samples", color="C1")
plt.tight_layout(); plt.show()

## 4. The reverse trajectory

Watch the noise blob reorganize into a spiral. Compare it with notebook 01's
forward plots — this is the same journey, run backwards.

In [ ]:
_, traj = my_ddpm_sample(model, schedule, (2000, 2), device, return_trajectory=True)
picks = [0, 40, 80, 120, 160, 200]
fig, axes = plt.subplots(1, len(picks), figsize=(2.6 * len(picks), 2.6))
for ax, i in zip(axes, picks):
    scatter_2d(ax, traj[i], f"t = {max(schedule.timesteps - i, 0)}", color="C1")
plt.suptitle("Your reverse process: noise -> swiss roll")
plt.tight_layout(); plt.show()

## 🎉 You built a diffusion model

Let's name what you derived and implemented:

| Piece | The idea |
|---|---|
| **Forward process** | shrink + add noise, variance preserved |
| **Nice property** | $x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\varepsilon$ — jump to any $t$ in $O(1)$ |
| **ε-prediction** | knowing the noise ⟺ knowing the reverse mean |
| **Simple loss** | the ELBO collapses to plain MSE on $\varepsilon$ |
| **Ancestral sampler** | iterate the posterior mean + noise, from $x_T$ to $x_0$ |

Every ingredient of a modern image generator is here. What changes for real images
is mostly the **network** (U-Net instead of MLP) — the mathematics you just
derived stays the same.

**Part 2 next:** a minimal U-Net on MNIST, then DDIM for fast sampling and
classifier-free guidance for conditional generation.